# CDC-CP P1 — Stage 1 development runs on Colab

Runs the P1 learning-rate pair and the seed repeat, then applies the written selection and gate rules.

| Run | Config | What it is |
|---|---|---|
| **E1** | `configs/p1.yaml` | backbone LR 1e-5 (team default), seed 42 |
| **E2** | `configs/p1_lr1e-4.yaml` | backbone LR 1e-4, seed 42 |
| **E3** | the better of the two, seed 43 | measures seed noise (delta for the gate) |

**Round 2** (backbone 1e-4, 12 epochs; each variant changes one thing; every config runs with seeds 42 and 43):

| Run | Config | What it is |
|---|---|---|
| **B42**, **B43** | `configs/p1b_base.yaml` | control: 1e-4, 12 epochs, Baseline 1 augmentation |
| **A42**, **A43** | `configs/p1b_aug.yaml` | control + stronger augmentation |
| **L42**, **L43** | `configs/p1b_ls.yaml` | control + label smoothing 0.1 |

After all six finish, the last cell compares them (`scripts/p1_compare.py`).

Shared recipe: AdamW, heads 1e-3, weight decay 1e-4, batch 32, 20 epochs (4,300 updates), 1-epoch linear warm-up then cosine to zero, mixed precision, BatchNorm recomputed before each evaluation, Baseline 1's augmentation.

**Rules this notebook keeps**
- Only `DPD_Small_train.csv` and its 8,000 images are copied. The official test CSVs and images are never copied or read.
- No early stopping: every run trains its full budget. A person picks the checkpoint afterwards from `selection.md`.
- Everything that must survive a disconnect (checkpoints, `history.csv`, `last.pt`) is written to Google Drive.

**Before the first session:** on your PC run `python scripts/pack_for_colab.py`, then upload `cdc-cp_code.zip` to Drive. Put the DPD-Small data on Drive too: a folder holding `DPD_Small_train.csv` plus the image folder, or the `.zip` / `.7z` of it.

**After a disconnect or a GPU quota stop:** open the notebook later and choose **Run all**. Finished runs are skipped and the interrupted one resumes from its last finished epoch. Nothing needs editing.

## 1. Settings — edit these

In [ ]:
from pathlib import Path

DRIVE     = Path('/content/drive/MyDrive')
CODE_ZIP  = DRIVE / 'AML' / 'cdc-cp_code.zip'   # made by scripts/pack_for_colab.py
DATA_SRC  = DRIVE / 'DPD-Small'                 # folder with DPD_Small_train.csv + image folder, or a .zip / .7z of it
RUNS_DIR  = DRIVE / 'AML' / 'cdc-cp-runs'       # checkpoints, histories and decisions land here

RUN       = 'ROUND2'  # 'ROUND2' trains all six round-2 runs in a row | or one run: 'E1', 'E2', 'E3', 'B42', 'B43', 'A42', 'A43', 'L42', 'L43'
E3_CONFIG = None    # after E1 and E2 finish: 'configs/p1.yaml' or 'configs/p1_lr1e-4.yaml' (cell 13 says which)
WORKERS   = 2       # Colab gives 2 CPU cores
CHECKS    = False   # True runs the optional tests, timing and smoke cells (first session only)

CODE_DIR   = Path('/content/cdc-cp')
LOCAL_DATA = Path('/content/DPD_Small')
CACHE_DIR  = Path('/content/cache')

## 2. Mount Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Unpack the code
The code is replaced every session. Runs live on Drive, so nothing is lost.

In [ ]:
import os, shutil, subprocess, sys, zipfile

assert CODE_ZIP.is_file(), f'Upload the code zip first: {CODE_ZIP}'
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall('/content')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'pytest'], check=True)
os.chdir(CODE_DIR)
print(sorted(p.name for p in CODE_DIR.iterdir()))


def sh(*args):
    """Run a command in the repo and stream its output."""
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=dict(os.environ),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 4. Copy the training data to local disk (train CSV and its 8,000 images only)

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

src = DATA_SRC
if src.is_file():                       # an archive: extract once per session
    raw = Path('/content/dpd_raw')
    if not raw.exists():
        raw.mkdir()
        if src.suffix.lower() == '.zip':
            subprocess.run(['unzip', '-q', str(src), '-d', str(raw)], check=True)
        else:
            subprocess.run(['7z', 'x', '-y', f'-o{raw}', str(src)], check=True, stdout=subprocess.DEVNULL)
    src = raw

train_csv = next(p for p in src.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

# The image folder is whichever folder under the CSV's folder holds the first training file.
first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} under {train_csv.parent}'

dst = LOCAL_DATA / 'DPD_Small_dataset'
dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, LOCAL_DATA / 'DPD_Small_train.csv')


def copy_one(name):
    s, d = image_src / name, dst / name
    if not (d.exists() and d.stat().st_size == s.stat().st_size):
        shutil.copy2(s, d)


with ThreadPoolExecutor(16) as pool:
    list(pool.map(copy_one, train.filename))
missing = [n for n in train.filename if not (dst / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(LOCAL_DATA)
print(f'8,000 training images in {dst}; copied from {image_src}')

## 5. Shared overrides for every command

In [ ]:
COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']
RUNS = {
    'E1': ('configs/p1.yaml', []),
    'E2': ('configs/p1_lr1e-4.yaml', []),
    'E3': (E3_CONFIG, ['seed=43']),
    'B42': ('configs/p1b_base.yaml', []), 'B43': ('configs/p1b_base.yaml', ['seed=43']),
    'A42': ('configs/p1b_aug.yaml', []),  'A43': ('configs/p1b_aug.yaml', ['seed=43']),
    'L42': ('configs/p1b_ls.yaml', []),   'L43': ('configs/p1b_ls.yaml', ['seed=43']),
}
print(COMMON)

## 6. Build and verify the image cache
Local disk, a few minutes. Files that already exist and load correctly are kept.

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 7. (Optional, CHECKS = True) Unit tests on CPU, about 2 minutes


In [ ]:
if CHECKS:
    sh(sys.executable, '-m', 'pytest', '-q', 'tests/test_model_bn_loss.py', 'tests/test_evaluate.py', 'tests/test_train.py')
else:
    print('skipped (CHECKS = False)')


## 8. (Optional, CHECKS = True) G0 timing: how long one P1 run takes here


In [ ]:
if CHECKS:
    sh(sys.executable, 'scripts/time_run.py', '--config', 'configs/p1.yaml', '--workers', WORKERS, '--steps', 60, f'paths.cache_dir={CACHE_DIR}')
else:
    print('skipped (CHECKS = False)')


## 9. (Optional, CHECKS = True) Smoke test: one real epoch into a throwaway folder


In [ ]:
if CHECKS:
    shutil.rmtree('/content/smoke', ignore_errors=True)
    sh(sys.executable, 'scripts/train.py', 'configs/p1.yaml', *COMMON, 'name=smoke', 'paths.runs_dir=/content/smoke', '--max-epochs', 1)
else:
    print('skipped (CHECKS = False)')


## 10. Train
With `RUN = 'ROUND2'` this trains B42, B43, A42, A43, L42, L43 one after another. Finished runs print `nothing to do` and are skipped; an interrupted run resumes from its last finished epoch. If Colab stops (quota or disconnect), just open the notebook later and choose **Run all** again: nothing needs editing.


In [ ]:
ROUND2 = ['B42', 'B43', 'A42', 'A43', 'L42', 'L43']
TODO = ROUND2 if RUN == 'ROUND2' else [RUN]
for key in TODO:
    config, extra = RUNS[key]
    assert config, 'Set E3_CONFIG in cell 1 before running E3'
    print(f'===== {key}: {config} {" ".join(extra)} =====', flush=True)
    sh(sys.executable, 'scripts/train.py', config, *COMMON, *extra, '--resume')


## 11. Curves for every run so far

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def ma(h):
    return np.array([h[max(0, i - 1):i + 2].mean() for i in range(len(h))])


runs = sorted(p for p in RUNS_DIR.iterdir() if (p / 'history.csv').is_file())
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for p in runs:
    h = pd.read_csv(p / 'history.csv')
    line, = axes[0].plot(h.epoch, h.H, marker='o', ms=3, alpha=.4, label=f'{p.name} H')
    c = line.get_color()
    axes[0].plot(h.epoch, ma(h.H.to_numpy()), color=c, lw=2, label=f'{p.name} 3-epoch average')
    axes[1].plot(h.epoch, h.A_S, color=c, label=f'{p.name} seen')
    axes[1].plot(h.epoch, h.A_U, color=c, ls='--', label=f'{p.name} unseen')
    axes[2].plot(h.epoch, h.train_ce_disease, color=c, label=f'{p.name} train')
    axes[2].plot(h.epoch, h.val_ce_disease, color=c, ls='--', label=f'{p.name} validation (seen)')
for ax, title in zip(axes, ['Development H', 'Composition Top-1: seen solid, unseen dashed', 'Disease cross-entropy']):
    ax.set_title(title); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 12. Selection tables
One `selection.md` per run trained above. A person reads and signs each.


In [ ]:
import yaml
for key in TODO:
    config, extra = RUNS[key]
    name = yaml.safe_load(open(CODE_DIR / config))['name']
    run_dir = RUNS_DIR / f"{name}_seed{43 if 'seed=43' in extra else 42}"
    sh(sys.executable, 'scripts/select_table.py', run_dir)
    print(open(run_dir / 'selection.md', encoding='utf-8').read())


## 13. P1 round 1 decision (E1, E2, E3 only)
Gate E2 against E1, delta from E3, budget check. Skipped in ROUND2 mode. Writes `p1_decision.md` next to the runs.


In [ ]:
if RUN in ('E1', 'E2', 'E3'):
    e1, e2 = RUNS_DIR / 'p1_lr1e-5_seed42', RUNS_DIR / 'p1_lr1e-4_seed42'
    e3 = next((p for p in RUNS_DIR.glob('p1_lr*_seed43') if (p / 'history.csv').is_file()), None)
    args = ['--e1', e1, '--e2', e2] + (['--e3', e3] if e3 else [])
    sh(sys.executable, 'scripts/p1_decide.py', *args, '--out', RUNS_DIR / 'p1_decision.md')
else:
    print('skipped (round 1 only)')


## 14. Round 2 comparison
Compares `p1b_aug` and `p1b_ls` against `p1b_base` on seen pair accuracy and unseen disease accuracy. Skips until both control runs exist. Writes `p1_round2.md` next to the runs.


In [ ]:
if all((RUNS_DIR / f'p1b_base_seed{s}' / 'history.csv').is_file() for s in (42, 43)):
    sh(sys.executable, 'scripts/p1_compare.py', RUNS_DIR, '--out', RUNS_DIR / 'p1_round2.md')
else:
    print('Round 2 control (B42 and B43) not finished yet.')
